# 3D view of MRI, labels and predictions (vedo)

> **Archived**: true 3D viewing is not in use for now; see `archive/README.md`. Use `notebooks/view_slices.ipynb` / `compare_slices.ipynb` instead.

Each mask is drawn as solid surfaces: **orange = anterior, blue = posterior**. When a prediction is compared with the ground truth, the ground truth is a faint **white** surface around it. The MRI is shown as two grey planes, a floor below and a wall behind the hippocampus, so nothing hides the surfaces.

vedo builds the surfaces from the masks. **How they are shown** (`VIEW` in the setup cell):
- `"plotly"` (default): rotatable 3D inside the notebook. **Drag to rotate, scroll to zoom, right-drag to pan**; click a legend entry to hide or show it. Works in VS Code and JupyterLab with no extra install.
- `"window"`: a separate vedo window (drag to rotate, scroll to zoom, `q` to close it and continue).

Part 1 uses an MSD case; part 2 uses the external dataset crops made by `python -m hippo.external`. Research use only, not for diagnosis.

In [ ]:
VIEW = "plotly"  # "plotly": rotatable 3D in the notebook; "window": separate interactive vedo window

# Work from the project root (so runs/ and data/ resolve) and import hippo from src/
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "hippo").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "archive"))  # render3d.py is archived
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import nibabel as nib
import numpy as np
import plotly.graph_objects as go
import vedo
import yaml
from IPython.display import display

import render3d
from hippo import CLASS_NAMES, defaults, visualize

if VIEW == "window":
    vedo.settings.default_backend = "vtk"  # opens a real window instead of a notebook picture

OUT = Path("runs/figures/notebook")  # prediction masks are saved here
OUT.mkdir(parents=True, exist_ok=True)
print(f"vedo {vedo.__version__}, view: {VIEW}, project: {ROOT}")

In [ ]:
PARTS = {value: (part, render3d.COLORS[value]) for value, part in CLASS_NAMES.items()}  # anterior orange, posterior blue
# External labels are subfields. Names inferred by matching each label's centroid to the SPAM atlas maps
# (the dataset does not document them); colours chosen to differ from the prediction's orange/blue.
SUBFIELDS = {1: ("CA1-3", "gold"), 2: ("subiculum", "mediumseagreen"), 3: ("CA4-DG", "orchid")}


def predict(run, image, label=None, parts=True):
    """Save the run's prediction for `image` as NIfTI (aligned with the MRI) and print Dice if `label` is given.

    The image is preprocessed the way the run was trained (z-score or raw intensities).
    parts=False: compare the whole hippocampus only (for labels that are not anterior/posterior).
    """
    run = Path(run)
    normalize = yaml.safe_load((run / "config.yaml").read_text())["data"].get("normalize", "zscore")
    d = visualize.load(Path(image), Path(label) if label else None, normalize)
    mask = visualize.predict(run, d["image"])
    path = OUT / f"{Path(image).name.replace('.nii.gz', '')}_pred_{run.name}.nii.gz"
    nib.save(nib.Nifti1Image(mask.astype(np.uint8), np.asarray(d["image"].affine, dtype=np.float64)), path)
    if label:
        truth = d["label"][0].numpy().astype(int)
        whole = 2 * ((mask > 0) & (truth > 0)).sum() / max((mask > 0).sum() + (truth > 0).sum(), 1)
        per_part = f"anterior/posterior {visualize.dice(mask, truth)}, " if parts else ""
        print(f"{run.name:>13}: Dice {per_part}whole hippocampus {whole:.3f}")
    return path


def surfaces(path, labels=PARTS):
    """[(part, vedo mesh in world mm, colour)] for each label value of a NIfTI mask.

    labels: {value: (part name, colour)}, e.g. PARTS (anterior/posterior) or SUBFIELDS.
    """
    data, spacing, origin = render3d.load_ras(str(path))
    data = np.rint(data).astype(np.uint8)
    return [(part, vedo.Volume((data == value).astype(np.uint8), spacing=spacing, origin=origin)
             .isosurface(0.5).smooth().triangulate(), color)
            for value, (part, color) in labels.items() if (data == value).sum() >= 2]


def scene(image, solid=None, ghost=None):
    """`solid` mask in colour, optional `ghost` mask as a faint surface around it, and the MRI for context.

    solid / ghost: (name, path to a NIfTI mask) or (name, path, labels) with labels as in surfaces()
    (default PARTS). A ghost with anterior/posterior labels is drawn white so it does not look like
    the prediction; a ghost with other labels (e.g. SUBFIELDS) keeps its own colours.
    """
    layers = []  # (legend name, mesh, colour, opacity)
    if ghost:
        name, path, labels = ghost if len(ghost) == 3 else (*ghost, PARTS)
        layers += [(f"{name}: {p}", m, "white" if labels is PARTS else c, 0.3) for p, m, c in surfaces(path, labels)]
    if solid:
        name, path, labels = solid if len(solid) == 3 else (*solid, PARTS)
        layers += [(f"{name}: {p}", m, c, 1.0) for p, m, c in surfaces(path, labels)]
    if not layers:
        raise ValueError("nothing to show: give a solid and/or ghost mask")

    # MRI planes just below (axial) and just behind (coronal) everything shown
    b = np.array([m.bounds() for _, m, _, _ in layers])
    lo = b[:, 0::2].min(0)
    data, spacing, origin = render3d.load_ras(str(image))
    data = data.astype(np.float32)
    iz = int(np.clip((lo[2] - origin[2]) / spacing[2] - 2, 0, data.shape[2] - 1))
    iy = int(np.clip((lo[1] - origin[1]) / spacing[1] - 2, 0, data.shape[1] - 1))
    vmin, vmax = np.percentile(data, [1, 99.5])
    title = Path(image).name.replace(".nii.gz", "") + (f": {solid[0]}" if solid else "")

    if VIEW == "window":
        vol = vedo.Volume(data, spacing=spacing, origin=origin)
        planes = [p.cmap("gray", vmin=vmin, vmax=vmax).lighting("off") for p in (vol.zslice(iz), vol.yslice(iy))]
        meshes = [m.clone().c(c).alpha(a) for _, m, c, a in layers]
        vedo.show(*planes, *meshes, axes=1, bg="black", viewup="z", title=title, new=True).close()
        return None

    xs, ys, zs = (origin[a] + np.arange(data.shape[a]) * spacing[a] for a in range(3))
    flat = dict(colorscale="gray", cmin=vmin, cmax=vmax, showscale=False, hoverinfo="skip",
                lighting=dict(ambient=1.0, diffuse=0.0, specular=0.0))
    X, Y = np.meshgrid(xs, ys, indexing="ij")
    X2, Z2 = np.meshgrid(xs, zs, indexing="ij")
    traces = [
        go.Surface(x=X, y=Y, z=np.full_like(X, zs[iz]), surfacecolor=data[:, :, iz], name="MRI axial", **flat),
        go.Surface(x=X2, y=np.full_like(X2, ys[iy]), z=Z2, surfacecolor=data[:, iy, :], name="MRI coronal", **flat),
    ]
    for name, mesh, color, opacity in layers:
        v, f = mesh.vertices, np.asarray(mesh.cells)
        traces.append(go.Mesh3d(x=v[:, 0], y=v[:, 1], z=v[:, 2], i=f[:, 0], j=f[:, 1], k=f[:, 2],
                                color=color, opacity=opacity, name=name, showlegend=True, hoverinfo="name",
                                lighting=dict(ambient=0.45, diffuse=0.8, specular=0.25, roughness=0.6)))
    fig = go.Figure(traces)
    fig.update_layout(
        title=f"{title}  (drag: rotate, scroll: zoom, right-drag: pan, click legend: show/hide)",
        template="plotly_dark", height=650, margin=dict(l=0, r=0, t=40, b=0),
        scene=dict(aspectmode="data", xaxis_title="R (mm)", yaxis_title="A (mm)", zaxis_title="S (mm)",
                   # from the front, above and the side, so the MRI planes stay behind and below
                   camera=dict(eye=dict(x=-0.9, y=1.5, z=0.9), up=dict(x=0, y=0, z=1))),
    )
    return fig

## 1. MSD case: ground truth and the three runs

`hippocampus_135` is where `no_norm` fails (intensity group 2, raw values ~380,000); try `hippocampus_222` for a case where all runs work.

In [ ]:
CASE = "hippocampus_135"
RUNS = ["runs/baseline", "runs/clean_labels", "runs/no_norm"]

image, label = defaults.find_case(CASE, defaults.DATA_DIR)
scene(image, solid=("ground truth", label))

In [ ]:
# Predict with every run (a few seconds on CPU) and print Dice per run
preds = {Path(r).name: predict(r, image, label) for r in RUNS}

In [ ]:
# Each run's prediction in colour, ground truth as the white ghost around it
for name, path in preds.items():
    print(name)
    display(scene(image, solid=(name, path), ghost=("ground truth", label)))

## 2. External dataset (whole-brain MNI scans, subfield labels)

Needs the crops from `python -m hippo.external --src data/mri_dataset --run runs/baseline` (cases `s01_L`, `s01_R`, ...).

The labels here are subfields, shown in their own colours: **gold = CA1-3, green = subiculum, purple = CA4-DG** (names inferred from the SPAM atlas, not documented in the dataset). They are not anterior/posterior, so only the whole-hippocampus Dice is meaningful. In the comparison views the subfields are a faint coloured shell around the prediction (orange/blue).

In [ ]:
EXT_CASE = "s03_L"
ext_image = Path(f"data/external/imagesTr/{EXT_CASE}.nii.gz")
ext_label = Path(f"data/external/labelsTr/{EXT_CASE}.nii.gz")
assert ext_image.exists(), "Run python -m hippo.external first; it creates the crops"

display(scene(ext_image, solid=("ground truth", ext_label, SUBFIELDS)))
ext_preds = {Path(r).name: predict(r, ext_image, ext_label, parts=False) for r in RUNS}

In [ ]:
for name, path in ext_preds.items():
    print(name)
    display(scene(ext_image, solid=(name, path), ghost=("ground truth", ext_label, SUBFIELDS)))